# 1.0 Import libraries

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd

# 2.0 Data load

In [13]:
gas_average = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_09_29_gas_nutrition_and_categories_compilation/gas_clean_average_subsets_1234561h_2026_09_26.csv')
nutrition_average = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_09_29_gas_nutrition_and_categories_compilation/nutrition_average_1234561h_2026_09_26.csv')
grasses_categories = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_09_29_gas_nutrition_and_categories_compilation/grasses_categories.csv')
legumes_categories = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_09_29_gas_nutrition_and_categories_compilation/legumes_categories.csv')
shrubs_categories = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_09_29_gas_nutrition_and_categories_compilation/shrubs_categories.csv')

# 3.0 Gas and nutrition data merge

In [4]:
gas_average.columns

Index(['id_lab', 'id', 'subset', 'no', 'requisitioner', 'tax_name',
       'functional_group', 'net_gas_8h_ml', 'net_gas_24h_ml',
       'gas_ml_g_dm_incubated_24h', 'tddm', 'part_fact',
       'ch4_percentage_in_gas_8h', 'ch4_percentage_in_gas_24h',
       'ch4_ml_g_dm_incubated_24h', 'methane_intensity',
       'ch4_ml_g_ndf_digested_24h', 'n_replicates_gas'],
      dtype='object')

In [5]:
gas_average_2 = gas_average.drop(['id_lab', 'subset', 'no', 'requisitioner', 'tax_name', 'functional_group'], axis=1)

In [6]:
gas_nutrition = pd.merge(nutrition_average, gas_average_2, on="id", how="outer")

# 4.0 Calculating quartiles

In [27]:
def rank_and_quartiles(df):
    # Sort by tddm (descending) and ch4_intensity (ascending)
    df = df.sort_values(by=["TDDM", "CH4_Intensity"], ascending=[False, True]).reset_index(drop=True)
    # Add ascending ranking
    df.insert(0, "quartile_rank", range(1, len(df) + 1))
    # Assign quartiles based on ranking
    df["quartile"] = pd.qcut(df["quartile_rank"], q=4, labels=["Q1", "Q2", "Q3", "Q4"])
    return df

In [40]:
grasses_categories_2 = rank_and_quartiles(grasses_categories)
legumes_categories_2 = rank_and_quartiles(legumes_categories)
shrubs_categories_2 = rank_and_quartiles(shrubs_categories)

# 5.0 Adding functional group categories to the dataframe

In [41]:
# Extract useful columns
grasses_categories_3 = grasses_categories_2[['Accession_ID', 'CH4_Category',  'TDDM_Category', 'lmf_category', 'Overall_Rank', 'quartile','quartile_rank']]
legumes_categories_3 = legumes_categories_2[['Accession_ID', 'CH4_Category',  'TDDM_Category', 'lmf_categories', 'Overall_Rank', 'quartile','quartile_rank']]
shrubs_categories_3 = shrubs_categories_2[['Accession_ID', 'CH4_Category',  'TDDM_Category', 'lmf_categories', 'Overall_Rank', 'quartile','quartile_rank']]

In [42]:
# Rename columns
grasses_categories_3 = grasses_categories_3.rename(columns={"Accession_ID": "id", "Overall_Rank":"lmf_category_rank"})
legumes_categories_3 = legumes_categories_3.rename(columns={"Accession_ID": "id", "lmf_categories":"lmf_category", "Overall_Rank":"lmf_category_rank"})
shrubs_categories_3 = shrubs_categories_3.rename(columns={"Accession_ID": "id", "lmf_categories":"lmf_category", "Overall_Rank":"lmf_category_rank"})

# 6.0 Compiling data

In [58]:
categories = pd.concat([grasses_categories_3, legumes_categories_3, shrubs_categories_3], ignore_index=True)

In [68]:
gas_nutrition_categories = pd.merge(gas_nutrition, categories, on="id", how="outer")

In [69]:
gas_nutrition_categories.head(20)

,id_lab,id,subset,no,requisitioner,tax_name,functional_group,dm_percentage,ash_dm,om_percentage,...,ch4_ml_g_dm_incubated_24h,methane_intensity,ch4_ml_g_ndf_digested_24h,n_replicates_gas,CH4_Category,TDDM_Category,lmf_category,lmf_category_rank,quartile,quartile_rank
0,F25-2627,BR-02-1752-Cayman-Exc,3.0,209.0,Benchmark,Urochloa interespecific,Grass,96.64,17.21,82.79,...,28.08,51.03,246.95,9.0,Medium,Not High,category_2,125.0,Q1,90.0
1,F26-0564,BR-02-1752-Cayman-Opt,5.0,75.0,Breeding,Urochloa interespecific,Grass,94.41,13.11,86.89,...,27.33,48.08,0.80,6.0,Medium,Not High,category_2,77.0,Q1,51.0
2,F25-2628,BR-06-423-Cayman-Exc,3.0,211.0,Benchmark,Urochloa interespecific,Grass,96.67,10.43,89.57,...,27.90,50.20,242.70,9.0,Medium,Not High,category_2,109.0,Q1,78.0
3,F26-0565,BR-06-423-Cayman-Opt,5.0,77.0,Breeding,Urochloa interespecific,Grass,94.42,12.92,87.08,...,27.81,56.82,1.73,6.0,Medium,Not High,category_2,255.0,Q3,263.0
4,F25-2596,CIAT-10026,4.0,169.0,Genetic_bank,Stylosanthes seabrana,Herbaceous_legumes,92.75,9.97,90.03,...,20.95,51.61,0.52,7.0,Medium,Not High,category_2,182.0,Q3,300.0
5,F26-0507,CIAT-1011,5.0,NaN,Genetic_bank,Stylosanthes viscosa,Herbaceous_legumes,94.02,11.72,88.28,...,9.67,43.01,-0.22,6.0,Low,Not High,category_2,57.0,Q4,453.0
6,F26-0508,CIAT-1013,5.0,NaN,Genetic_bank,Stylosanthes fruticosa,Herbaceous_legumes,94.14,9.39,90.61,...,30.15,66.73,0.87,6.0,High,Not High,category_4,18.0,Q3,245.0
7,F24-3549,CIAT-10440,3.0,155.0,Genetic_bank,Stylosanthes humilis,Herbaceous_legumes,90.24,11.91,88.09,...,28.08,57.24,257.78,9.0,Medium,Not High,category_2,286.0,Q2,170.0
8,F24-3550,CIAT-10613,3.0,157.0,Genetic_bank,Stylosanthes scabra,Herbaceous_legumes,92.84,7.89,92.11,...,22.13,50.74,234.03,9.0,Medium,Not High,category_2,169.0,Q3,246.0
9,F24-3551,CIAT-10627,3.0,159.0,Genetic_bank,Stylosanthes scabra,Herbaceous_legumes,93.39,13.71,86.29,...,22.76,56.51,263.33,9.0,Medium,Not High,category_2,273.0,Q3,295.0


In [70]:
gas_nutrition_categories.to_csv('gas_nutrition_categories.csv', index=None)